In [11]:
import pandas as pd
import chromadb
import fastembed
import fastf1
from google import genai
print("All imports work ✅")
print(f"FastF1: {fastf1.__version__}")

All imports work ✅
FastF1: 3.8.3


In [13]:
#Second LLM and comparison
#Setup with dynamic paths

import os 
import json
import requests
import pandas as pd
from dotenv import load_dotenv

#Keeping the base path dynamic so it will work on any machine 
BASE = os.path.expanduser ('~/easyf1')

#loading api keys from .env 
load_dotenv(dotenv_path=os.path.join(BASE,'.env'))
API_KEY = os.getenv("GEMINI_API_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")

print(f"Base path; {BASE}")
print(f"Gemini key: {'Yes' if API_KEY else 'No'}")
print(f"OPENROUTER_KEY: {'Yes' if API_KEY else 'No'}")

Base path; /Users/user/easyf1
Gemini key: Yes
OPENROUTER_KEY: Yes


In [7]:
#Integrating embedding model + ChromaDB
import chromadb
from fastembed import TextEmbedding

#Local embedding model 
embedding_model = TextEmbedding(
    model_name= "BAAI/bge-small-en-v1.5",
    cache_dir= os.path.join(BASE, "model_cache")
)

def embed_text(text):
    """Turning text into vector via local fastembed"""
    return list (embedding_model.embed([text]))[0].tolist()

#connecting to multi season collection that was already built with chunks 
chroma_client = chromadb.PersistentClient(path=os.path.join(BASE, 'chromadb'))
collection = chroma_client.get_collection("easyf1_all_seasons")

print(f"Embedding model ready")
print(f"Collection: {collection.count()} chunks")

Embedding model ready
Collection: 6204 chunks


In [18]:
#OpenRouter caller
OPENROUTER_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

def call_openrouter(prompt, model=OPENROUTER_MODEL):
    """Sending a prompt to OpenRouter"""
    response = requests.post(
        url= "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_KEY}",
            "Content-Type" : "application/json"
        },
        data = json.dumps({
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        })
    )
    return response
#Quick liveness check  for  Nemotron still available to use
test = call_openrouter("Reply with exactly: OK")
print(f"Status: {test.status_code}")
if test.status_code == 200:
    print(f"Nemotron says: {test.json()['choices'][0]['message']['content']}")
    print("Challenger model is LIVE ")
else:
    print(f"Model issue: {test.text[:150]}")
    print("→ May have rotated out. We'll pick a new free model if so.")

Status: 200
Nemotron says: OK
Challenger model is LIVE 


In [ ]:
#Reloading key 
from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(BASE, '.env'))
API_KEY        = os.getenv("GEMINI_API_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

print(f"Gemini key:     {len(API_KEY) if API_KEY else 0} chars")
print(f"OpenRouter key: {len(OPENROUTER_KEY) if OPENROUTER_KEY else 0} chars")

Gemini key:     39 chars
OpenRouter key: 73 chars


In [19]:
#Gemini Caller 
from google import genai
gemini_client = genai.Client(api_key=API_KEY)

def ask_gemini(prompt):
    """sendinfg a prompt to Gemini Flash Lite 2.5"""
    response = gemini_client.models.generate_content(
        model = "gemini-2.5-flash-lite",
        contents= prompt
    )
    return response.text

#testing 
test = ask_gemini("Reply back with: OK")
print(f"Gemini says: {test}")
print("Gemini good to go")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini says: OK
Gemini good to go


In [21]:
#Retrieval and shared prompt 
def retrieve_context (scenario, n=6):
    """Retrieving the top n-most relevant chunks for a scenario"""
    q_emb = embed_text(scenario)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=n, 
        include=['documents', 'metadatas']
    )
    return results ['documents'][0]

def build_prompt(scenario, context_chunks):
    """Building one prompt used identically by both modes"""
    context_text = "\n" .join(f"-{c}" for c in context_chunks)
    prompt = f"""You are an expart F1 race strategist. Using only the historical 
            context provided, give a strategy recommendation

HISTORICAL_CONTEXT FROM PAST RACES:
{context_text}

SCENARIO
{scenario}

Provide your recommendation in this exact format:
VERDICT: (one clear recommendation in a sentence)
CONFIDENCE: (High / Medium / Low)
REASONING: (2-3 sentences grounded in the historical context above)
RISK: (the main risk of this recommendation)"""
    return prompt

#Test scenario
sample = "It's lap 30 of 52 at Silverstone. Hamilton is P4 on 30-lap-old- had tyres. A safety car has just been deployed, should he pit?"
chunks = retrieve_context(sample)

print (f"Retrieved {len(chunks)} for the safety car")
for i, c in enumerate(chunks, 1):
    print(f"{i}. {c[:195]}...")




Retrieved 6 for the safety car
1. PIT STOP: Lap 33 of 52, 2023 British Grand Prix at Silverstone. Lewis Hamilton (HAM) racing for Mercedes made a pit stop from position 3. Switched from 33-lap-old MEDIUM tyres to SOFT tyres, unde...
2. PIT STOP: Lap 39 of 52, 2022 British Grand Prix at Silverstone. Lewis Hamilton (HAM) racing for Mercedes made a pit stop from position 3. Switched from 6-lap-old HARD tyres to SOFT tyres, under a...
3. SAFETY CAR PERIOD: 2023 British Grand Prix at Silverstone. A Safety Car was deployed on lap 33 and lasted until lap 38 (6 laps total, out of 52). This created a strategic pit window where teams c...
4. SAFETY CAR PERIOD: 2022 British Grand Prix at Silverstone. A Safety Car was deployed on lap 38 and lasted until lap 42 (5 laps total, out of 52). This created a strategic pit window where teams c...
5. SAFETY CAR PERIOD: 2021 British Grand Prix at Silverstone. A Safety Car was deployed on lap 1 and lasted until lap 2 (2 laps total, out of 52). This created a

In [24]:
#Dual Model comparison with same context
import time

def ask_nemotron(prompt):
    """Sending prompt to Nemotron via OpenRouter"""
    resp = call_openrouter(prompt)
    if resp.status_code == 200:
        return resp.json()['choices'][0]['message']['content']
    else:
        return f"[Error {resp.status_code}: {resp.text[:100]}]"

def compare_models(scenario, n=6, show_context=False):
    """Running One scenario through both models with identical prompts
        Returns both recommendations for comparisons"""
    #Retrieve context once, both models with same chunks
    chunks = retrieve_context(scenario, n=n)
    prompt = build_prompt(scenario, chunks)

    print("="*65)
    print("SCENARIO")
    print("="*65)
    print(scenario)
    print()

    if show_context:
        print("Context retrieved")
        for c in chunks:
            print(f" - {c[:85]}...")
        print()

    # Model 1: Gemini
    print("="*65)
    print("GEMINI 2.5 Flash-Lite (Google, proprietary)")
    print("="*65)
    gemini_answer = ask_gemini(prompt)
    print(gemini_answer)
    print()

    # Model 2: Nemotron
    print("="*65)
    print("NEMOTRON 3 Ultra 550B (NVIDIA, open-weight)")
    print("="*65)
    nemotron_answer = ask_nemotron(prompt)
    print(nemotron_answer)
    print()

    return {
        'scenario': scenario,
        'context': chunks,
        'gemini': gemini_answer,
        'nemotron': nemotron_answer,
    }


print("Comparison function ready ✅")

Comparison function ready ✅


In [25]:
#Initiating the comparison

scenario_1 = (
    "It's lap 32 of 50 at the 2023 British Grand Prix(Silverstone)"
    "Lewis Hamilton is running P4 on 30 laps old hard tyres."
    "A safety car has just been deployed after an incident. The pit lane is open"
    "Should hamilton pit now for fresh tyres, or stay out? "
)

result_1 = compare_models(scenario_1, show_context=True)

SCENARIO
It's lap 32 of 50 at the 2023 British Grand Prix(Silverstone)Lewis Hamilton is running P4 on 30 laps old hard tyres.A safety car has just been deployed after an incident. The pit lane is openShould hamilton pit now for fresh tyres, or stay out? 

Context retrieved
 - PIT STOP: Lap 33 of 52, 2023 British Grand Prix at Silverstone. Lewis Hamilton (HAM) ...
 - STINT: 2022 British Grand Prix at Silverstone. Lewis Hamilton (HAM, Mercedes) ran sti...
 - PIT STOP: Lap 39 of 52, 2022 British Grand Prix at Silverstone. Lewis Hamilton (HAM) ...
 - STINT: 2022 British Grand Prix at Silverstone. Lewis Hamilton (HAM, Mercedes) ran sti...
 - STINT: 2021 British Grand Prix at Silverstone. Lewis Hamilton (HAM, Mercedes) ran sti...
 - PIT STOP: Lap 33 of 52, 2022 British Grand Prix at Silverstone. Lewis Hamilton (HAM) ...

GEMINI 2.5 Flash-Lite (Google, proprietary)
VERDICT: Hamilton should pit now for fresh tyres during the Safety Car.
CONFIDENCE: High
REASONING: In 2023, Hamilton pitted unde

In [ ]:

# Test scenarios 

scenarios = [
    # 1. Safety car pit decision (already tested — the classic)
    "It's lap 32 of 52 at the 2023 British Grand Prix (Silverstone). "
    "Lewis Hamilton is P4 on 30-lap-old hard tyres. A Safety Car has just "
    "been deployed. Should he pit now or stay out?",

    # 2. Undercut opportunity (no safety car — pure strategy)
    "It's lap 20 of 57 at the 2022 Hungarian Grand Prix. A driver is P5, "
    "2 seconds behind the P4 car, both on used medium tyres. Fresh mediums "
    "are available. Is this a good moment to attempt an undercut?",

    # 3. Tyre longevity gamble (one-stop vs two-stop)
    "It's lap 25 of 44 at the 2021 French Grand Prix. A driver is P2 on "
    "20-lap-old hard tyres with no safety car. They could try to reach the "
    "end on these tyres (a one-stop) or pit again for a two-stop. Which is wiser?",

    # 4. Defending position late (track position vs fresh rubber)
    "It's lap 45 of 53 at the 2023 Austrian Grand Prix. A driver is P3 on "
    "18-lap-old soft tyres being caught by a rival on fresher tyres. Should "
    "they pit for fresh tyres and give up track position, or defend on old tyres?",

    # 5. Wet-to-dry crossover (changing conditions)
    "It's lap 15 of 44 at a wet race. The track is drying and a driver is P6 "
    "on intermediate tyres. Slick tyres are getting faster. When should they "
    "switch to slicks?",
]

print(f"{len(scenarios)} scenarios ready to run.")
for i, s in enumerate(scenarios, 1):
    print(f"\n{i}. {s[:80]}...")

5 scenarios ready to run.

1. It's lap 32 of 52 at the 2023 British Grand Prix (Silverstone). Lewis Hamilton i...

2. It's lap 20 of 57 at the 2022 Hungarian Grand Prix. A driver is P5, 2 seconds be...

3. It's lap 25 of 44 at the 2021 French Grand Prix. A driver is P2 on 20-lap-old ha...

4. It's lap 45 of 53 at the 2023 Austrian Grand Prix. A driver is P3 on 18-lap-old ...

5. It's lap 15 of 44 at a wet race. The track is drying and a driver is P6 on inter...


In [ ]:

# Testing all scenarios through both models
import time

all_results = []

for i, scenario in enumerate(scenarios, 1):
    print(f"\n{'#'*65}")
    print(f"# SCENARIO {i} of {len(scenarios)}")
    print(f"{'#'*65}")
    
    result = compare_models(scenario, show_context=False)
    result['scenario_num'] = i
    all_results.append(result)
    
    # Pause between scenarios to respect rate limits
    if i < len(scenarios):
        print("\n[pausing 5s before next scenario...]")
        time.sleep(5)

print(f"\n\n{'='*65}")
print(f"ALL {len(all_results)} SCENARIOS COMPLETE ✅")
print(f"{'='*65}")


#################################################################
# SCENARIO 1 of 5
#################################################################
SCENARIO
It's lap 32 of 52 at the 2023 British Grand Prix (Silverstone). Lewis Hamilton is P4 on 30-lap-old hard tyres. A Safety Car has just been deployed. Should he pit now or stay out?

GEMINI 2.5 Flash-Lite (Google, proprietary)
VERDICT: Pit now under the Safety Car.
CONFIDENCE: High
REASONING: Historically, Lewis Hamilton has successfully utilized Safety Cars for advantageous pit stops at Silverstone, such as switching to Softs on lap 33 in 2023 from P3 and switching to Softs on lap 39 in 2022 from P3. Pitting now under the Safety Car allows for a free tire change, similar to the 2023 scenario where he entered from P3 and switched to softer tires for the remainder of the race.
RISK: The risk is that other cars may also pit, negating the strategic advantage of fresher tires, or that a red flag situation could occur, nullifying the be

In [ ]:
# More robust Nemotron caller
import time

def ask_nemotron(prompt, retries=2):
    """Send prompt to Nemotron via OpenRouter, with safety checks."""
    for attempt in range(retries + 1):
        resp = call_openrouter(prompt)
        
        if resp.status_code == 200:
            data = resp.json()
            # Check the response actually has choices
            if 'choices' in data and len(data['choices']) > 0:
                return data['choices'][0]['message']['content']
            else:
                # 200 but no choices — likely transient, retry
                if attempt < retries:
                    print(f"   [empty response, retrying in 5s... attempt {attempt+1}]")
                    time.sleep(5)
                    continue
                else:
                    return f"[NO CHOICES in response: {str(data)[:150]}]"
        elif resp.status_code == 429:
            # Rate limited — wait longer
            if attempt < retries:
                print(f"   [rate limited, waiting 15s... attempt {attempt+1}]")
                time.sleep(15)
                continue
            else:
                return "[RATE LIMITED after retries]"
        else:
            return f"[ERROR {resp.status_code}: {resp.text[:100]}]"
    
    return "[FAILED after all retries]"

print("Robust ask_nemotron ready ✅")

Robust ask_nemotron ready ✅


In [31]:

# Saving the comparison results
import json
import pickle
from datetime import datetime

# Verifying the results folder exists
results_dir = os.path.join(BASE, 'results')
os.makedirs(results_dir, exist_ok=True)

# Timestamp for this comparison run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Saving as JSON
json_path = os.path.join(results_dir, f'comparison_{timestamp}.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

# Saving as pickle exact Python backup
pkl_path = os.path.join(results_dir, f'comparison_{timestamp}.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Saved {len(all_results)} comparison results ✅")
print(f"  JSON:   {json_path}")
print(f"  Pickle: {pkl_path}")

# Also recording which models were used
metadata = {
    'timestamp': timestamp,
    'model_a': 'gemini-2.5-flash-lite',
    'model_b': OPENROUTER_MODEL,
    'num_scenarios': len(all_results),
    'knowledge_base': 'easyf1_all_seasons',
    'num_chunks': collection.count(),
    'retrieval_n': 6,
}
meta_path = os.path.join(results_dir, f'comparison_{timestamp}_meta.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"  Metadata: {meta_path}")
print()
print("Run metadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

Saved 5 comparison results ✅
  JSON:   /Users/user/easyf1/results/comparison_20260907_175705.json
  Pickle: /Users/user/easyf1/results/comparison_20260907_175705.pkl
  Metadata: /Users/user/easyf1/results/comparison_20260907_175705_meta.json

Run metadata:
  timestamp: 20260907_175705
  model_a: gemini-2.5-flash-lite
  model_b: nvidia/nemotron-3-ultra-550b-a55b:free
  num_scenarios: 5
  knowledge_base: easyf1_all_seasons
  num_chunks: 6204
  retrieval_n: 6


In [32]:
# Testing more scenarios

scenarios_batch2 = [
    # 6. Early safety car, hard call on whether to pit from the lead
    "It's lap 8 of 44 at the 2021 Emilia Romagna Grand Prix. A driver leads "
    "on medium tyres when a Safety Car is deployed. Pitting is cheap now but "
    "commits them to a long final stint. Should the leader pit?",

    # 7. Overcut (opposite of undercut — staying out to gain)
    "It's lap 30 of 53 at the 2022 Austrian Grand Prix. A driver is P3 on "
    "aging mediums. The car ahead has just pitted onto fresh tyres. Should "
    "our driver pit immediately to cover, or stay out and attempt an overcut?",

    # 8. Deep into a stint, deciding whether to extend
    "It's lap 40 of 70 at the 2022 Monaco Grand Prix. A driver is P4 on "
    "22-lap-old hard tyres that still feel decent. Track position is king at "
    "Monaco. Should they extend this stint as long as possible or pit soon?",

    # 9. Double-stack risk (teammates pitting together)
    "It's lap 25 of 58 at the 2023 Spanish Grand Prix. A Safety Car is out. "
    "A driver is P5 and their teammate is P6, right behind. Both need tyres. "
    "Should the team double-stack both cars in the same pit window?",

    # 10. Chasing with a big tyre delta late
    "It's lap 48 of 57 at the 2023 Hungarian Grand Prix. A driver is P4, "
    "12 seconds behind P3, but on tyres 15 laps fresher. Is it worth pushing "
    "hard to chase, or is P4 the realistic ceiling?",

    # 11. Very early stop reacting to damage/graining
    "It's lap 6 of 52 at the 2021 Spanish Grand Prix. A driver started on "
    "softs which are already graining badly in P7. Should they pit very early "
    "for mediums and accept an off-sequence strategy?",

    # 12. Late safety car gamble from a safe position
    "It's lap 50 of 56 at the 2022 Dutch Grand Prix. A driver is comfortably "
    "P3 on old hards when a late Safety Car appears. Pitting for softs risks "
    "position but offers attack pace. Should they gamble?",

    # 13. Circuit where overtaking is easy (strategy less critical)
    "It's lap 30 of 53 at the 2023 Italian Grand Prix (Monza). A driver is "
    "P6 on old mediums. Overtaking is easy at Monza. Should they pit for "
    "fresh tyres and rely on overtaking to recover position?",

    # 14. Protecting a lead with a rival threatening undercut
    "It's lap 22 of 44 at the 2021 French Grand Prix. A driver leads but the "
    "P2 car is within undercut range and likely to pit next lap. Should the "
    "leader pit first to defend, or stay out and risk the undercut?",

    # 15. Thin-data scenario (tests knowledge base limits)
    "It's lap 35 of 71 at the 2021 Monaco Grand Prix. A driver is P8 on "
    "20-lap-old mediums in clean air. There's no safety car and overtaking is "
    "nearly impossible. What's the optimal strategy from here?",
]

print(f"Batch 2: {len(scenarios_batch2)} more scenarios ready.")
for i, s in enumerate(scenarios_batch2, 6):
    print(f"\n{i}. {s[:75]}...")

Batch 2: 10 more scenarios ready.

6. It's lap 8 of 44 at the 2021 Emilia Romagna Grand Prix. A driver leads on m...

7. It's lap 30 of 53 at the 2022 Austrian Grand Prix. A driver is P3 on aging ...

8. It's lap 40 of 70 at the 2022 Monaco Grand Prix. A driver is P4 on 22-lap-o...

9. It's lap 25 of 58 at the 2023 Spanish Grand Prix. A Safety Car is out. A dr...

10. It's lap 48 of 57 at the 2023 Hungarian Grand Prix. A driver is P4, 12 seco...

11. It's lap 6 of 52 at the 2021 Spanish Grand Prix. A driver started on softs ...

12. It's lap 50 of 56 at the 2022 Dutch Grand Prix. A driver is comfortably P3 ...

13. It's lap 30 of 53 at the 2023 Italian Grand Prix (Monza). A driver is P6 on...

14. It's lap 22 of 44 at the 2021 French Grand Prix. A driver leads but the P2 ...

15. It's lap 35 of 71 at the 2021 Monaco Grand Prix. A driver is P8 on 20-lap-o...


In [33]:
# Cell 12: Run batch 2 through both models

import time

batch2_results = []

for idx, scenario in enumerate(scenarios_batch2):
    scenario_num = idx + 6  # continue numbering from 6
    print(f"\n{'#'*65}")
    print(f"# SCENARIO {scenario_num} of 15")
    print(f"{'#'*65}")
    
    result = compare_models(scenario, show_context=False)
    result['scenario_num'] = scenario_num
    batch2_results.append(result)
    
    if idx < len(scenarios_batch2) - 1:
        print("\n[pausing 6s before next scenario...]")
        time.sleep(6)

print(f"\n\n{'='*65}")
print(f"BATCH 2 COMPLETE: {len(batch2_results)} scenarios ✅")
print(f"{'='*65}")


#################################################################
# SCENARIO 6 of 15
#################################################################
SCENARIO
It's lap 8 of 44 at the 2021 Emilia Romagna Grand Prix. A driver leads on medium tyres when a Safety Car is deployed. Pitting is cheap now but commits them to a long final stint. Should the leader pit?

GEMINI 2.5 Flash-Lite (Google, proprietary)
VERDICT: The leader should pit under the Safety Car.
CONFIDENCE: High
REASONING: The historical context shows multiple drivers, including those in midfield positions, pitting under Safety Cars around lap 32 (which is mid-race) to switch to fresh tyres. Pitting now, even with a potentially long final stint, offers a significant advantage in tyre performance for the remainder of the race.
RISK: The main risk is that the long final stint on fresh tyres may not be enough to overcome the time lost by pitting and the potential for the Safety Car to end early, leaving the leader vulnerable.



In [34]:

# Merging both test scenarios

import json
import pickle
from datetime import datetime

# Merging both batches into one complete dataset
full_dataset = all_results + batch2_results
full_dataset.sort(key=lambda x: x['scenario_num'])  # order 1-15

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = os.path.join(BASE, 'results')

# Saving complete set — JSON + pickle
json_path = os.path.join(results_dir, f'comparison_full15_{timestamp}.json')
with open(json_path, 'w') as f:
    json.dump(full_dataset, f, indent=2)

pkl_path = os.path.join(results_dir, f'comparison_full15_{timestamp}.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(full_dataset, f)

# Metadata
metadata = {
    'timestamp': timestamp,
    'model_a': 'gemini-2.5-flash-lite',
    'model_b': OPENROUTER_MODEL,
    'num_scenarios': len(full_dataset),
    'knowledge_base': 'easyf1_all_seasons',
    'num_chunks': collection.count(),
    'retrieval_n': 6,
}
meta_path = os.path.join(results_dir, f'comparison_full15_{timestamp}_meta.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Full dataset saved: {len(full_dataset)} scenarios ✅")
print(f"  JSON:   {json_path}")
print(f"  Pickle: {pkl_path}")
print(f"  Meta:   {meta_path}")
print()
print(f"Scenario numbers: {[r['scenario_num'] for r in full_dataset]}")

Full dataset saved: 15 scenarios ✅
  JSON:   /Users/user/easyf1/results/comparison_full15_20260907_184605.json
  Pickle: /Users/user/easyf1/results/comparison_full15_20260907_184605.pkl
  Meta:   /Users/user/easyf1/results/comparison_full15_20260907_184605_meta.json

Scenario numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
